# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# Optionally, suppress SettingWithCopyWarning for chained assignment in pandas
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We'll list the record sets, then examine their fields and columns, referencing all by their `@id`.

In [ ]:
# List all available record sets by @id with their fields and columns
def get_record_sets(ds):
    record_sets = []
    if hasattr(ds.metadata, 'record_sets'):
        for rset in ds.metadata.record_sets:
            name = getattr(rset, 'name', None)
            id_ = getattr(rset, '@id', None)
            record_sets.append({'@id': id_, 'name': name, 'fields': [], 'columns': []})
            # Add fields by @id
            if hasattr(rset, 'fields'):
                for f in rset.fields:
                    field_id = getattr(f, '@id', None)
                    record_sets[-1]['fields'].append(field_id)
            # Add columns by @id
            if hasattr(rset, 'columns'):
                for c in rset.columns:
                    col_id = getattr(c, '@id', None)
                    record_sets[-1]['columns'].append(col_id)
    return record_sets

record_sets = get_record_sets(dataset)

if not record_sets:
    print("No explicit record sets are defined in metadata. Listing distributions as available data objects.")
    # Some datasets provide all data through distributions/files only
    if hasattr(metadata, 'distributions') or hasattr(metadata, 'distribution'):
        distributions = getattr(metadata, 'distributions', None) or getattr(metadata, 'distribution', None)
        for d in distributions:
            dist_id = getattr(d, '@id', None)
            print("Distribution @id:", dist_id)
    else:
        print("No record sets nor distributions available.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"Record Set: {rs['name']} (@id: {rs['@id']})")
        print(f"  Fields @id: {rs['fields']}")
        print(f"  Columns @id: {rs['columns']}")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
If no explicit record sets are present, we extract from each distribution.

In [ ]:
# Gather all available record sets or file objects for extraction by @id

# Get the list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        try:
            # Using @id to select the record set
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded DataFrame for Record Set @id: {record_set_id}, shape = {df.shape}")
            else:
                print(f"No records found for Record Set @id: {record_set_id}")
        except Exception as e:
            print(f"Error loading records for Record Set @id: {record_set_id}: {e}")
else:
    print("No record sets found. Attempting to extract tabular data from each distribution.")
    # Try to extract data from distributions (data files)
    distributions = getattr(metadata, 'distributions', None) or getattr(metadata, 'distribution', None)
    if distributions:
        for d in distributions:
            dist_id = getattr(d, '@id', None)
            try:
                records = list(dataset.records(distribution=dist_id))
                if records:
                    df = pd.DataFrame(records)
                    dataframes[dist_id] = df
                    print(f"Loaded DataFrame for Distribution @id: {dist_id}, shape = {df.shape}")
                else:
                    print(f"No records found for Distribution @id: {dist_id}")
            except Exception as e:
                print(f"Error loading records for Distribution @id: {dist_id}: {e}")
    else:
        print("No distributions available to extract data.")

# Display the first DataFrame's columns and preview, if available
if dataframes:
    first_key = list(dataframes.keys())[0]
    print(f"\nColumns for first data object (@id: {first_key}):")
    print(dataframes[first_key].columns.tolist())
    display(dataframes[first_key].head())
else:
    print("No tabular data loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We will select a numeric field/column by its `@id` (or column name if available), filter the DataFrame, normalize values, and, when possible, group by a categorical field.

In [ ]:
# Select an example DataFrame and fields for EDA
if dataframes:
    df_key = list(dataframes.keys())[0]  # Use the first available data object
    df = dataframes[df_key]
    print(f"EDA on DataFrame loaded from @id: {df_key}")
    print("Available columns:", df.columns.tolist())

    # Attempt to select a numeric column automatically
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field and len(df.columns) > 0:
        # Try to coerce first column to numeric
        try:
            df[df.columns[0]] = pd.to_numeric(df[df.columns[0]], errors='coerce')
            if pd.api.types.is_numeric_dtype(df[df.columns[0]]):
                numeric_field = df.columns[0]
        except Exception:
            pass

    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        # Remove missing values for this field
        df_numeric = df[numeric_field].dropna()
        if len(df_numeric) > 0:
            threshold = df_numeric.mean()  # Use mean as threshold for demo
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean):")
            print(filtered_df.head())

            # Normalize
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - df_numeric.mean()) / df_numeric.std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a categorical column
            group_field = None
            for col in df.columns:
                if col != numeric_field and df[col].dtype == object:
                    group_field = col
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field, dropna=True)[numeric_field].mean().reset_index()
                print(f"Grouped by {group_field}, mean {numeric_field}:")
                print(grouped_df.head())
            else:
                print("No suitable categorical group field available.")
        else:
            print("No valid (non-missing) numeric values for analysis.")
    else:
        print("No numeric field detected in DataFrame. EDA cannot proceed.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and, if available, its breakdown by a categorical group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field and numeric_field in df.columns:
    # Histogram of numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group field, if available
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable data available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load and explore a FAIR^2 data package defined by a Croissant schema using `mlcroissant`.
- Inspect metadata, discover available record sets and their unique `@id`s.
- Extract records into pandas DataFrames for further analysis.
- Carry out basic EDA and visualize numeric field distributions.

> You can now adapt this workflow to any Croissant-defined dataset—remember to reference your selected entities by their `@id` for maximum reproducibility and clarity.